# Bankruptcy cascade — **networkx** version (R replica)

Same model as `bankruptcy_cascade_python_replica.ipynb`, but the graph operations use **networkx**
(`DiGraph`), which mirrors igraph 1:1:

| igraph (R) | networkx |
|---|---|
| `neighbors(G, i, 'out')` | `G.successors(i)` / `G.succ[i]` |
| `neighbors(G, i, 'in')` | `G.predecessors(i)` / `G.pred[i]` |
| `degree(G, i, 'all')` | `G.degree(i)` |
| `strength(G, ., weights)` | `G.degree(weight='weight')` |
| `E(G)$weight[edge j->i]` | `G[j][i]['weight']` |

Only `method='node'` is used (per-node cascade, as in `ER_10.R`). It is validated node-for-node
against the R outputs and spot-checked against the fast dict version for the 2.7 network.

> Note: networkx is slower than the dict/array version. For the dense `2_7` network use the
> dict notebook for the full per-node run; this one is the readable, igraph-faithful reference.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import networkx as nx

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
R_DIR = PROJECT_ROOT / 'financial-contagion-in-R'
NETWORK_SIZE = 10000
KI_BASE, KI_TARGET = 0.04, 0.10
print('R dir:', R_DIR)

R dir: /Users/rubenmarques/Documents/Repositórios/Thesis/financial-contagion-in-R


## Build the networkx DiGraph

In [2]:
def build_graph(edges_csv, network_size=NETWORK_SIZE):
    e = pd.read_csv(edges_csv)
    n = max(int(network_size), int(e[['from', 'to']].max().max()))
    G = nx.DiGraph()
    G.add_nodes_from(range(1, n + 1))                      # include isolated nodes (ids 1..n)
    G.add_weighted_edges_from(
        zip(e['from'].astype(int), e['to'].astype(int), e['weight'].astype(float))
    )
    return G, n

## `judge_bankrupt` + `simulate_bankrupt` (networkx, exact port)

In [3]:
def judge_bankrupt(G, this_batch_neighbor, already_bankrupt, target_policy=False, deg_q95=None):
    Ki = KI_BASE
    bankrupt = []
    for i in this_batch_neighbor:
        if target_policy and G.degree(i) >= deg_q95:
            Ki = KI_TARGET   # not reset per-iteration, matching judge_bankrupt.R
        s = 0.0; has_bankrupt_in = False
        for j, attr in G.pred[i].items():          # in-neighbors j -> i
            if j in already_bankrupt:
                s += attr['weight']; has_bankrupt_in = True
        if has_bankrupt_in and s > Ki:
            bankrupt.append(i)
    return set(bankrupt)


def simulate_bankrupt(G, method='node', type='banks', target_policy=False,
                      initial_node=None, deg_q95=None, rng=None):
    if method == 'biggest':
        initial = max(G.nodes(), key=lambda v: G.degree(v))   # first max in node order (1..n)
    elif method == 'node':
        if initial_node is None:
            raise ValueError("initial_node required when method='node'")
        initial = int(initial_node)
    else:
        initial = (rng or random).choice(list(G.nodes()))

    bankrupt = {initial}
    this_batch = [initial]
    number_bankrupt = 1
    while True:
        cand = set()
        for b in this_batch:
            cand.update(G.succ[b])                  # out-neighbors b -> *
        new_b = judge_bankrupt(G, cand, bankrupt, target_policy, deg_q95)
        bankrupt |= new_b
        if len(bankrupt) == number_bankrupt:
            break
        number_bankrupt = len(bankrupt)
        this_batch = list(new_b)
    return len(bankrupt) if type == 'num' else sorted(bankrupt)

## Per-node driver (mirrors `ER_10.R`)

In [4]:
def compute_node_cascades(edges_csv, avg_degree, network_size=NETWORK_SIZE, target_policy=False):
    G, n = build_graph(edges_csv, network_size)
    deg_q95 = float(np.quantile([d for _, d in G.degree()], 0.95))
    cascade = [simulate_bankrupt(G, method='node', initial_node=node, type='num',
                                 target_policy=target_policy, deg_q95=deg_q95)
               for node in range(1, n + 1)]
    ind, outd, totd = dict(G.in_degree()), dict(G.out_degree()), dict(G.degree())
    sin = dict(G.in_degree(weight='weight')); sout = dict(G.out_degree(weight='weight'))
    stot = dict(G.degree(weight='weight'))
    ids = list(range(1, n + 1))
    df = pd.DataFrame({
        'avg_degree': avg_degree, 'node_id': ids,
        'degree_in': [ind[i] for i in ids], 'degree_out': [outd[i] for i in ids],
        'degree_total': [totd[i] for i in ids],
        'strength_in': [sin[i] for i in ids], 'strength_out': [sout[i] for i in ids],
        'strength_total': [stot[i] for i in ids], 'cascade_size': cascade,
    })
    df['cascade_percentage'] = df['cascade_size'] / network_size
    return df

## (1) Validate node-for-node against R outputs (small networks)

In [5]:
for label, avg in [('0_2', 0.2), ('0_4', 0.4), ('0_6', 0.6), ('0_8', 0.8)]:
    ref_p = R_DIR / f'network_er_avgdeg_{label}_nodes_with_cascade.csv'
    edg_p = R_DIR / f'network_er_avgdeg_{label}_edges.csv'
    if not (ref_p.exists() and edg_p.exists()):
        print(f'{label}: missing, skip'); continue
    ref = pd.read_csv(ref_p)
    got = compute_node_cascades(edg_p, avg)
    m = ref[['node_id', 'cascade_size']].merge(got[['node_id', 'cascade_size']],
                                                on='node_id', suffixes=('_R', '_nx'))
    nmatch = (m['cascade_size_R'] == m['cascade_size_nx']).sum()
    print(f'{label}: {nmatch}/{len(m)} match  (max|diff|={(m.cascade_size_R-m.cascade_size_nx).abs().max()})')

0_2: 10000/10000 match  (max|diff|=0)


0_4: 10000/10000 match  (max|diff|=0)


0_6: 10000/10000 match  (max|diff|=0)


0_8: 10000/10000 match  (max|diff|=0)


## (2) Spot-check vs the fast dict version on `2_7` (50 random nodes)

In [6]:
dict_csv = R_DIR / 'network_er_avgdeg_2_7_nodes_with_cascade.csv'
if dict_csv.exists():
    G27, n27 = build_graph(R_DIR / 'network_er_avgdeg_2_7_edges.csv')
    q95 = float(np.quantile([d for _, d in G27.degree()], 0.95))
    ref = pd.read_csv(dict_csv).set_index('node_id')['cascade_size']
    rng = random.Random(0)
    sample = rng.sample(range(1, n27 + 1), 50)
    ok = all(simulate_bankrupt(G27, method='node', initial_node=v, type='num', deg_q95=q95) == int(ref[v])
             for v in sample)
    print('2_7 spot-check (50 nodes): networkx == dict version ->', ok)
else:
    print('Run the dict notebook first to produce the 2_7 reference.')

2_7 spot-check (50 nodes): networkx == dict version -> True


## (Optional) Full per-node run on `2_7` with networkx

Slower than the dict version; enable only if you specifically want the networkx path to write it.

In [7]:
RUN_FULL_2_7 = False
if RUN_FULL_2_7:
    df27 = compute_node_cascades(R_DIR / 'network_er_avgdeg_2_7_edges.csv', avg_degree=2.7)
    df27.to_csv(R_DIR / 'network_er_avgdeg_2_7_nodes_with_cascade.csv', index=False)
    print('cascade min/mean/max =', df27.cascade_size.min(), round(df27.cascade_size.mean(), 3), df27.cascade_size.max())
else:
    print('RUN_FULL_2_7 = False (use the dict notebook for the fast full run).')

RUN_FULL_2_7 = False (use the dict notebook for the fast full run).
